In [1]:
import pandas as pd
import os
import kagglehub

path = kagglehub.dataset_download("zusmani/rainfall-in-pakistan")
print("Path to dataset files:", path)
# Load dataset
path = "/kaggle/input/datasets/zusmani/rainfall-in-pakistan"
file_path = os.path.join(path, os.listdir(path)[0])

df = pd.read_csv(file_path)

# Cleaning
df.columns = df.columns.str.strip()
df = df.rename(columns={'Rainfall - (MM)': 'rainfall'})
df['Month'] = pd.to_datetime(df['Month'], format='%B').dt.month
df['date'] = pd.to_datetime(df[['Year','Month']].assign(DAY=1))
df = df.sort_values('date')

Path to dataset files: /kaggle/input/datasets/zusmani/rainfall-in-pakistan


In [2]:
# Rolling averages
df['rain_3month_avg'] = df['rainfall'].rolling(3).mean()
df['rain_6month_avg'] = df['rainfall'].rolling(6).mean()

In [3]:
# Lag features
df['lag1'] = df['rainfall'].shift(1)
df['lag2'] = df['rainfall'].shift(2)

In [4]:
# Spike feature
df['spike'] = df['rainfall'] / (df['rain_3month_avg'] + 1)


In [5]:
# Drop NA
df = df.dropna()


In [6]:
# Target creation
low = df['rainfall'].quantile(0.33)
high = df['rainfall'].quantile(0.66)

def classify(x):
    if x < low:
        return 0
    elif x < high:
        return 1
    else:
        return 2

df['risk_level'] = df['rainfall'].apply(classify)

print("Features Created:")
df.head()

Features Created:


,rainfall,Year,Month,date,rain_3month_avg,rain_6month_avg,lag1,lag2,spike,risk_level
5,12.88130,1901,6,1901-06-01,21.826700,23.953333,38.3046,14.2942,0.564308,1
6,68.09300,1901,7,1901-07-01,39.759633,28.564533,12.8813,38.3046,1.670599,2
7,16.58360,1901,8,1901-08-01,32.519300,29.278100,68.0930,12.8813,0.494748,1
8,13.33910,1901,9,1901-09-01,32.671900,27.249300,16.5836,68.0930,0.396149,1
9,3.10757,1901,10,1901-10-01,11.010090,25.384862,13.3391,16.5836,0.258747,0
